### Import packages, define paths and load data

In [1]:
import pandas as pd
from pathlib import Path
import json
import numpy as np

In [ ]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/processed"

sci_fi_data_path = DATA_DIR / "sci_fi_stories_text_descriptors.jsonl"
romance_data_path = DATA_DIR / "romance_stories_text_descriptors.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_text_descriptors.jsonl"

In [3]:
def load_jsonl_data(file_path):
    with open(file_path, 'r') as f:
        stories = [json.loads(line) for line in f]
    return stories

In [4]:
lit_fiction_stories = load_jsonl_data(literary_fiction_data_path)
romance_stories = load_jsonl_data(romance_data_path)
sci_fi_stories = load_jsonl_data(sci_fi_data_path)

In [5]:
# combine all stories into one list for easier processing
all_stories = lit_fiction_stories + sci_fi_stories + romance_stories

### Calculating averages

In [6]:
relevant_metrics = ["first_order_coherence", # how well each sentence connects to the previous one. Closer to 1 = smoother flow.
    "second_order_coherence", # coherence across larger spans (e.g., paragraphs). Higher values = more globally coherent text.
    "lix", # readability - Higher is more difficult. 20–30 is easy; 40–50 is difficult.
    "dependency_distance_mean", # coherence/structure - Average “distance” between syntactically linked words. Higher = more complex sentences.
    "mean_word_length", # mean length of words; higher = more complex vocabulary.
    "proportion_unique_tokens", # Lexical diversity; higher = more varied vocabulary.
    "pos_prop_ADJ", # % of adjectives
    "pos_prop_NOUN", # % of nouns
    "pos_prop_VERB", # % of verbs
    "pos_prop_PRON", # % of pronouns
    "duplicate_ngram_chr_fraction_5", # repetitiveness - High value → story repeats the same phrases or chunks of text often
    "oov_ratio", # fraction of out-of-vocabulary words
    "word_count" # number of words in the story
]

In [7]:
def calculate_average_metrics(stories, metrics):
    df = pd.DataFrame(stories)
    agg_df = df.groupby("genre")[metrics].agg(["mean", "std"]).reset_index()
    
    # Flatten the multi-level column names (e.g. "metric_mean", "metric_std")
    agg_df.columns = ["genre"] + [f"{metric}_{stat}" for metric, stat in agg_df.columns[1:]]
    
    return agg_df

In [8]:
df = calculate_average_metrics(all_stories, relevant_metrics)

In [9]:
df

,genre,first_order_coherence_mean,first_order_coherence_std,second_order_coherence_mean,second_order_coherence_std,lix_mean,lix_std,dependency_distance_mean_mean,dependency_distance_mean_std,mean_word_length_mean,...,pos_prop_VERB_mean,pos_prop_VERB_std,pos_prop_PRON_mean,pos_prop_PRON_std,duplicate_ngram_chr_fraction_5_mean,duplicate_ngram_chr_fraction_5_std,oov_ratio_mean,oov_ratio_std,word_count_mean,word_count_std
0,literary fiction,0.847183,0.018026,0.840788,0.018685,33.557676,2.908240,2.537066,0.127764,3.878460,...,0.129803,0.007076,0.103022,0.011891,0.013355,0.009638,0.013164,0.004695,1744.440172,214.587650
1,romance,0.845533,0.018018,0.839229,0.018177,34.049052,2.936062,2.534847,0.126325,3.910832,...,0.131761,0.007219,0.109756,0.012833,0.011597,0.009042,0.015314,0.005078,1543.187576,189.074505
2,science fiction,0.806736,0.019885,0.796591,0.021021,33.770968,2.971703,2.363125,0.107254,3.983514,...,0.128938,0.007092,0.093973,0.012594,0.014850,0.009556,0.019348,0.004562,1882.950714,232.331440
